In [ ]:
import torch
torch.__version__
import triton
import triton.language as tl

In [ ]:
from RMSnorm_triton_kernel import *

In [ ]:
class TritonRMSNorm(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, w, eps=1e-6):
        M, N = x.shape
        y = torch.empty_like(x)
        rstd = torch.zeros(M, dtype=torch.float32, device=x.device)

        # Launch fwd kernel
        BLOCK = triton.next_power_of_2(N)
        grid = (M,)
        rmsnorm_fwd_kernel[grid](x, w, y, rstd, x.stride(0), N, eps, BLOCK)

        # save necessary context for backwardpass (req. for memory efficiency)
        ctx.save_for_backward(x, w, rstd)
        ctx.BLOCK = BLOCK
        ctx.N = N
        return y

    @staticmethod
    def backward(ctx, dy): #dy is grad_output
        x, weight, rstd = ctx.saved_tensors
        M, N = x.shape

        dx = torch.empty_like(x)
        dw = torch.zeros_like(weight) # Look at the triton_bwd_kernel,
                                        # dw is accumulated using atomic_add
                                        # So use zeroes_like instead of empty_like
        # Launch bwd kernel
        rmsnorm_bwd_kernel[(M,)](dy, x, w, rstd, dx, dw, x.stride(0),
                                 dy.stride(0), M, N, BLOCK_SIZE=ctx.BLOCK)
        return dx, dw, None # eps is a const and has no grad

In [ ]:
# Pytorch verification
def torch_rmsnorm(x, weight, eps=1e-6):
    # Perform math in float32 for numerical stability, matching the kernel logic
    x_f32 = x.to(torch.float32)
    # RMS = sqrt(mean(x^2) + eps)
    rms = torch.sqrt(torch.mean(x_f32**2, dim=-1, keepdim=True) + eps)
    x_norm = x_f32 / rms
    # Scale by weight and cast back to original dtype
    return (x_norm * weight.to(torch.float32)).to(x.dtype)

In [ ]:
# # Usage - works with autograd
torch.manual_seed(12)
x = torch.randn(1024, 768, device='cuda', dtype=torch.float16, requires_grad=True)
w = torch.ones(768, device='cuda', dtype=torch.float16, requires_grad=True)
y = TritonRMSNorm.apply(x, w)
y.sum().backward() # dx and dw computed via Triton kernels

# Clone inputs for the reference pass to ensure identical starting points
x_ref = x.detach().clone().requires_grad_(True)
w_ref = w.detach().clone().requires_grad_(True)

# 2. Run Triton Version
y_triton = TritonRMSNorm.apply(x, w)
loss_triton = y_triton.sum()
loss_triton.backward()

# 3. Run PyTorch Reference Version
y_ref = torch_rmsnorm(x_ref, w_ref)
loss_ref = y_ref.sum()
loss_ref.backward()

# 4. Compare Results
# We use atol (absolute tolerance) and rtol (relative tolerance)
# because float16 has limited precision.
def compare(name, triton_val, ref_val):
    if torch.allclose(triton_val, ref_val, atol=1e-3, rtol=1e-3):
        print(f"✅ {name} matches!")
    else:
        diff = (triton_val - ref_val).abs().max()
        print(f"❌ {name} differs! Max diff: {diff:.6f}")

compare("Forward (y)", y_triton, y_ref)
compare("Gradient (dx)", x.grad, x_ref.grad)
compare("Gradient (dw)", w.grad, w_ref.grad)